# Make LLM Helpers Auditable and Reproducible

In this lesson, we keep the same helper pattern from earlier modules, then add logging and versioning configs so every run is reviewable and reproducible.


## 1 — Setup

We use the same setup pattern as the first helper notebook: install libraries, load credentials, and read the dataset.


In [1]:
#%pip install -qq google-genai pandas scikit-learn matplotlib seaborn python-dotenv

In [ ]:
import os
import json
from datetime import datetime

from google import genai
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
df = pd.read_csv("../data/hr_analytics.csv")


## 2 — The LLM Helper Function from Module 0

In [ ]:
SYSTEM_PROMPT = (
    "Write pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. Avoid deprecated arguments or methods. "
    "Store the final result in `result_df`. "
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations."
)

In [ ]:
def eda_helper(question, frame=df, show_code: bool = False):
    """Ask a plain-English question about df; get back a DataFrame."""
    prompt = f"Columns: {list(frame.columns)}\nQuestion: {question}"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": SYSTEM_PROMPT,
        },
    )

    code = response.text
    if "```" in code:
        code = code.split("```")[1].replace("python", "").strip()

    # Optionally print the generated code before executing it.
    # This lets you inspect what the model wrote and verify that it matches your intent.
    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---\n")

    env = {"pd": pd, "df": frame.copy()}
    exec(code, env, env)

    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result


## 3 — Enterprise Essentials: Auditability, Reuse, Versioning

A production helper needs more than a correct answer. It needs:

| Concern | Why it matters |
|---|---|
| **Auditability** | Regulators and stakeholders must see *what code* produced the result |
| **Reuse** | Anyone should be able to re-run the same analysis and get the same results |
| **Versioning** | Track which model and prompt version produced each result |

Below we wrap the same helper with a lightweight audit log. 

### 3.1 — Keep a Simple Log

Once you can see the code, the next natural question is: *can I go back and check what ran earlier?*

A lightweight log with just a list of `(timestamp, question, code)` entries that gives you a basic audit trail without any extra complexity.

In [ ]:
log = []

LOG_CONFIG = {
    "version": "1.0.0",
    "model": "gemini-2.5-flash",
    "system_instruction": SYSTEM_PROMPT,
    "temperature": 0.0,
    "seed": 42,
}


def eda_helper(question, helper_config=LOG_CONFIG, frame=df, show_code=False):
    prompt = f"Columns: {list(frame.columns)}\nQuestion: {question}"

    response = client.models.generate_content(
        model=helper_config["model"],
        contents=prompt,
        config={
            "temperature": helper_config["temperature"],
            "seed": helper_config["seed"],
            "system_instruction": helper_config["system_instruction"],
        },
    )

    code = response.text
    if "```" in code:
        code = code.split("```")[1].replace("python", "").strip()

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---")

    entry = {
        "timestamp": datetime.now().isoformat(),
        "agent_version": helper_config["version"],
        "model": helper_config["model"],
        "question": question,
        "prompt": prompt,
        "system_instruction": helper_config["system_instruction"],
        "temperature": helper_config["temperature"],
        "seed": helper_config["seed"],
        "generated_code": code,
        "status": "pending",
    }

    try:
        env = {"pd": pd, "df": frame.copy()}
        exec(code, env, env)
        result = env.get("result_df")
        if result is None:
            raise RuntimeError("Generated code did not assign `result_df`.")
        entry["status"] = "success"
        entry["result_shape"] = str(result.shape)
    except Exception as e:
        entry["status"] = "error"
        entry["error"] = str(e)
        log.append(entry)
        print(f"Execution failed: {e}")
        return None

    log.append(entry)
    return result

In [ ]:
eda_helper("What are the top 10 highest paid employees?", frame=df, show_code=True)

In [ ]:
log

### 3.2 — Inspect the Audit Log

In [ ]:
audit_df = pd.DataFrame(log)
print(f"Queries logged : {len(audit_df)}")
if len(audit_df):
    print(f"Successful     : {(audit_df['status'] == 'success').sum()}")
    print(f"Failed         : {(audit_df['status'] == 'error').sum()}")
    display(audit_df[["timestamp", "question", "status", "model", "generated_code"]])
else:
    print("No log entries yet. Run eda_helper(...) first.")

### 3.3 — Replay Without an API Call

Since the generated code is stored, you can re-run any past analysis offline — no LLM call required. This saves cost and reuses the exact same generated logic. To reproduce the exact same result, you also need the same data and package versions.

In [ ]:
def replay(query_index):
    """Re-execute a logged query without calling the LLM."""
    entry = log[query_index]
    print(f"Replaying : {entry['question']}")
    print(f"Originally: {entry['timestamp']}")
    print(f"Code:\n{entry['generated_code']}")

    env = {"pd": pd, "df": df.copy()}
    exec(entry["generated_code"], env, env)
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result


replay(0)

### 3.4 — Version & Compare Models

Change the model or prompt and the log tells you exactly which config produced each result.

In [ ]:
# CONFIG_V2 shows how you'd test a different model — swap the model name to compare outputs.
CONFIG_V2 = {
    **LOG_CONFIG,
    "version": "2.0.0",
    "model": "gemini-2.0-flash",
}

question = "Who are the top 5 employees closest to retirement age (65) who have never been promoted?"

result_v1 = eda_helper(question, helper_config=LOG_CONFIG)
result_v2 = eda_helper(question, helper_config=CONFIG_V2)

print(f"=== v1 ({LOG_CONFIG['model']}) ===")
print(result_v1)
print(f"\n=== v2 ({CONFIG_V2['model']}) ===")
print(result_v2)

### 3.5 — Export the Audit Log

In [ ]:
with open("log.json", "w") as f:
    json.dump(log, f, indent=2)
print("\u2713 Audit log exported to log.json")